# DSM - Example 5 (hour 2 sub-problem)

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

Hour-2 follow-up problem for the e5 series.

In [1]:
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint, SolverFactory,
    minimize, value
)

# ---- Data: loaded from external file 'DSM_IC_e5_hr2_data.txt' ----
import sys, pathlib
# Make the ampl_data parser importable (one level up from this notebook)
_pkg = pathlib.Path.cwd().parent
if str(_pkg) not in sys.path: sys.path.insert(0, str(_pkg))
from ampl_data import parse_ampl_data

_d = parse_ampl_data('DSM_IC_e5_hr2_data.txt')

GEN_data     = _d['GEN']
PERIOD_data  = _d['PERIOD']
gen_min      = _d['gen_min']
gen_max      = _d['gen_max']
gen_RRlimit  = _d['gen_RRlimit']
gen_OpCost   = _d['gen_OpCost']
gen_NlCost   = _d['gen_NlCost']
gen_SuCost   = _d['gen_SuCost']
gen_Init     = _d['gen_Init']
Time_TotalPd = _d['Time_TotalPd']

DSM_d = 40  # deferred load carried over from hr1 (the original AMPL .mod had DSM_d = 0, which was a bug)

m = ConcreteModel()
m.GEN    = Set(initialize=GEN_data, ordered=True)
m.PERIOD = Set(initialize=PERIOD_data, ordered=True)

m.gen_min     = Param(m.GEN, initialize=gen_min)
m.gen_max     = Param(m.GEN, initialize=gen_max)
m.gen_RRlimit = Param(m.GEN, initialize=gen_RRlimit)
m.gen_OpCost  = Param(m.GEN, initialize=gen_OpCost)
m.gen_NlCost  = Param(m.GEN, initialize=gen_NlCost)
m.gen_Init    = Param(m.GEN, initialize=gen_Init)
m.Time_TotalPd = Param(m.PERIOD, initialize=Time_TotalPd)

m.Pg = Var(m.GEN, m.PERIOD)

m.obj = Objective(
    rule=lambda mm: sum(mm.gen_OpCost[g]*mm.Pg[g,t] + mm.gen_NlCost[g]
                       for g in mm.GEN for t in mm.PERIOD),
    sense=minimize
)

m.PowerBalance = Constraint(m.PERIOD,
    rule=lambda mm,t: sum(mm.Pg[g,t] for g in mm.GEN) == mm.Time_TotalPd[t] + DSM_d)
m.genLimit_Min = Constraint(m.GEN, m.PERIOD, rule=lambda mm,g,t: mm.gen_min[g] <= mm.Pg[g,t])
m.genLimit_Max = Constraint(m.GEN, m.PERIOD, rule=lambda mm,g,t: mm.Pg[g,t] <= mm.gen_max[g])
m.genRR_Up     = Constraint(m.GEN, m.PERIOD, rule=lambda mm,g,t: mm.Pg[g,t]-mm.gen_Init[g] <= mm.gen_RRlimit[g])
m.genRR_Dn     = Constraint(m.GEN, m.PERIOD, rule=lambda mm,g,t: mm.gen_Init[g]-mm.Pg[g,t] <= mm.gen_RRlimit[g])

model = m

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
m = model
print("g  t   Pg")
for g in m.GEN:
    for t in m.PERIOD:
        print(f"{g}  {t}   {value(m.Pg[g,t]):.3f}")

Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmpl7y131yl.pyomo.lp


Reading time = 0.00 seconds
x1: 9 rows, 3 columns, 10 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]


Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90


MIPGap  0



Optimize a model with 9 rows, 3 columns and 10 nonzeros
Model fingerprint: 0x7ce47889
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]


...

Presolve removed 9 rows and 3 columns


Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time


       0    1.9500000e+03   0.000000e+00   0.000000e+00      0s



Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.950000000e+03


ok optimal
g  t   Pg
1  1   75.000
2  1   35.000
